# 00 — Live Loop Control Panel

**Purpose**: Start, stop, and monitor the live data collection loop from inside the notebook.

All cells run the deterministic engine with domain-aware tool filtering.  
`ENABLE_EVOLUTION = False` — no LLM calls, no tool generation, no registry mutations.

**Sections**:
1. Environment check (config, tools, SQLite)
2. Single-shot dry run (one poll, no writes)
3. External data collection (FRED, BLS, Weather, Odds)
4. Start background loop (threaded, interruptible)
5. Stop background loop
6. Status snapshot
7. Tail recent runs from SQLite

## 1. Environment Check

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import (
    ENABLE_EVOLUTION,
    ENABLE_MOCK_TOOLS,
    USE_LOGISTIC_CALIBRATION,
    SCORING_MODE,
    SQLITE_ENABLED,
    SQLITE_DB_FILE,
    OUTPUTS_DIR,
    EXTERNAL_DIR,
    WATCHER_POLL_INTERVAL_SEC,
)
from tools.registry import build_default_registry

registry = build_default_registry()

print('═' * 60)
print('  ENVIRONMENT CHECK')
print('═' * 60)
print(f'ENABLE_EVOLUTION           : {ENABLE_EVOLUTION}')
print(f'ENABLE_MOCK_TOOLS          : {ENABLE_MOCK_TOOLS}')
print(f'USE_LOGISTIC_CALIBRATION   : {USE_LOGISTIC_CALIBRATION}')
print(f'SCORING_MODE               : {SCORING_MODE}')
print(f'SQLITE_ENABLED             : {SQLITE_ENABLED}')
print(f'SQLITE_DB_FILE             : {SQLITE_DB_FILE}')
print(f'OUTPUTS_DIR                : {OUTPUTS_DIR}')
print(f'EXTERNAL_DIR               : {EXTERNAL_DIR}')
print(f'Default poll interval      : {WATCHER_POLL_INTERVAL_SEC}s')
print()
print(f'Registered tools ({len(registry.tool_names)}):')
for name in registry.tool_names:
    tag = ' [generated]' if name in registry.generated_tool_names else ''
    print(f'  • {name}{tag}')

assert not ENABLE_EVOLUTION, 'STOP: Evolution must be OFF for data collection mode.'
print()
print('✓ Safe to run live loop.')

## 2. Single-Shot Dry Run (New System)

Runs the **full pipeline** with domain-aware tool filtering:
- Market classification (sports/economics/politics/weather/crypto)
- Domain-filtered tool selection
- probability_edge scoring for sports markets
- Market sanity checks (abstain on price <= 0.01)

**Nothing is written to disk** — just prints results.

In [ ]:
from api.kalshi_client import KalshiClient
from engine.market_classifier import classify_market, get_domain_tools_with_weather
from engine.tool_runner import run_tools
from engine.scorer import compute_score
from schemas import EventInput, FormulaSpec, ToolSelection
from datetime import datetime, timezone

# --- Configuration -----------------------------------------------------------
MARKET_FILTER = None      # e.g. 'NBA' to restrict to NBA markets
THRESHOLD     = 0.55
MAX_MARKETS   = 5         # keep small for a quick check
# -----------------------------------------------------------------------------

client = KalshiClient()
registry = build_default_registry()

try:
    markets = client.get_active_markets(limit=MAX_MARKETS)
except Exception as e:
    print(f'API error: {e}')
    markets = []

if MARKET_FILTER:
    markets = [m for m in markets if MARKET_FILTER.lower() in m.get('market_id','').lower()]

print(f'Fetched {len(markets)} markets\n')

for i, mkt in enumerate(markets, 1):
    market_id = mkt['market_id']
    title = mkt.get('title', 'Unknown')
    price = mkt.get('last_price', 0.0)
    
    # Classify domain
    domain = classify_market(market_id, title)
    
    # Filter tools to domain
    all_tool_names = registry.tool_names
    domain_tool_names = get_domain_tools_with_weather(domain, all_tool_names, market_id, title)
    
    # Build equal-weight formula with domain-filtered tools
    n = len(domain_tool_names)
    if n == 0:
        print(f'[{i}] {market_id}: No tools for domain={domain}, skipping')
        continue
        
    weight = 1.0 / n
    selections = [ToolSelection(tool_name=name, tool_inputs={}, weight=weight) for name in domain_tool_names]
    formula = FormulaSpec(
        selections=selections,
        aggregation='weighted_sum',
        threshold=THRESHOLD,
        rationale=f'Equal-weight {n} {domain} tools'
    )
    
    # Build event
    event = EventInput(
        event_id=mkt.get('event_id', market_id),
        market_id=market_id,
        market_title=title,
        current_price=price,
        timestamp=datetime.now(timezone.utc)
    )
    
    # Market sanity check
    from main import _market_is_valid
    valid, reason = _market_is_valid(event)
    if not valid:
        print(f'[{i}] {market_id}: ABSTAIN ({reason})')
        continue
    
    # Run tools
    tool_outputs, statuses = run_tools(event, formula, registry)
    
    # Score (use probability_edge for sports, else signal_sum)
    scoring_mode = 'probability_edge' if domain == 'sports' else SCORING_MODE
    score = compute_score(tool_outputs, formula, scoring_mode=scoring_mode, current_market_price=price)
    
    bet_flag = '[BET]' if score.bet_triggered else '     '
    print(f'{bet_flag} [{i}] {market_id:<30s}  domain={domain:<10s}  price={price:.3f}  score={score.final_score:.4f}')
    print(f'      Tools: {domain_tool_names}')
    if scoring_mode == 'probability_edge' and score.model_probability:
        print(f'      p_model={score.model_probability:.3f}  edge={score.edge:.4f}  z={score.raw_score_z:.3f}')
    print()

## 3. External Data Collection

Run external data collectors to populate JSONL files for external tools.

**Collectors:**
- FRED (macro indicators)
- BLS (labor/CPI)
- WeatherAPI (3-day forecast for NBA arena cities)
- TheOddsAPI (NBA/NFL sportsbook lines)

Run this **before starting the live loop** to ensure external tools have fresh data.

In [ ]:
from prediction_agent.collector.external.fred_collector import collect_fred_data
from prediction_agent.collector.external.bls_collector import collect_bls_data
from prediction_agent.collector.external.weather_collector import collect_weather_data
from prediction_agent.collector.external.odds_collector import collect_odds_data

print('═' * 60)
print('  EXTERNAL DATA COLLECTION')
print('═' * 60)

# FRED (macro indicators)
print('\n[1/4] FRED (macro indicators)...')
try:
    fred_count = collect_fred_data()
    print(f'  ✓ Collected {fred_count} FRED data points')
except Exception as e:
    print(f'  ✗ FRED collection failed: {e}')

# BLS (labor/CPI)
print('\n[2/4] BLS (labor/CPI)...')
try:
    bls_count = collect_bls_data()
    print(f'  ✓ Collected {bls_count} BLS data points')
except Exception as e:
    print(f'  ✗ BLS collection failed: {e}')

# Weather (3-day forecast)
print('\n[3/4] WeatherAPI (NBA arena cities)...')
try:
    weather_count = collect_weather_data()
    print(f'  ✓ Collected {weather_count} weather forecasts')
except Exception as e:
    print(f'  ✗ Weather collection failed: {e}')

# Odds (sportsbook lines)
print('\n[4/4] TheOddsAPI (NBA/NFL lines)...')
try:
    odds_count = collect_odds_data()
    print(f'  ✓ Collected {odds_count} sportsbook odds')
except Exception as e:
    print(f'  ✗ Odds collection failed: {e}')

print('\n' + '═' * 60)
print('  EXTERNAL DATA COLLECTION COMPLETE')
print('═' * 60)

## 4. Start Background Loop

Runs `run_live_loop()` in a daemon thread so the notebook stays responsive.  
Set `DRY_RUN = True` to collect data without writing, or `False` to write to SQLite + JSONL.

> **Stop**: Run **Section 5** or interrupt the kernel (`Kernel > Interrupt`).

In [ ]:
import threading
import prediction_agent.live.run_live_loop as _loop_mod
from prediction_agent.live.run_live_loop import run_live_loop

# --- Configuration -----------------------------------------------------------
POLL_INTERVAL = 60      # seconds between Kalshi API polls
MARKET_FILTER = None    # e.g. 'NBA' to restrict to NBA markets
THRESHOLD     = 0.55
MAX_MARKETS   = 50
DRY_RUN       = False   # set True to run without writing to disk
# -----------------------------------------------------------------------------

# Reset shutdown flag so a fresh loop can start
_loop_mod._SHUTDOWN = False

def _loop_target():
    run_live_loop(
        poll_interval=POLL_INTERVAL,
        market_filter=MARKET_FILTER,
        dry_run=DRY_RUN,
        threshold=THRESHOLD,
        max_markets=MAX_MARKETS,
    )

_loop_thread = threading.Thread(target=_loop_target, daemon=True, name='live-loop')
_loop_thread.start()

print(f'Live loop started (thread: {_loop_thread.name}, poll_interval={POLL_INTERVAL}s, dry_run={DRY_RUN})')
print('Run Section 5 to stop gracefully.')

## 5. Stop Background Loop

In [ ]:
import prediction_agent.live.run_live_loop as _loop_mod

_loop_mod._SHUTDOWN = True
print('Shutdown signal sent.')

# Wait up to 5s for the thread to notice
if '_loop_thread' in dir():
    _loop_thread.join(timeout=5)
    status = 'stopped' if not _loop_thread.is_alive() else 'still running (will exit after current iteration)'
    print(f'Loop thread: {status}')
else:
    print('No loop thread found in this session.')

## 6. Status Snapshot

Equivalent to `python -m prediction_agent.live.status` — all data from SQLite.

In [ ]:
from prediction_agent.live.status import _collect_status, print_status

metrics = _collect_status()
print_status(metrics)

## 7. Tail Recent Runs from SQLite

In [ ]:
import pandas as pd
from config import SQLITE_DB_FILE

N_ROWS = 20  # how many recent runs to show

if SQLITE_DB_FILE.exists():
    from prediction_agent.storage.sqlite_store import SQLiteStore
    store = SQLiteStore()

    rows = store.query(
        'SELECT run_id, timestamp, market_id, score, threshold, bet_triggered, outcome '
        'FROM runs ORDER BY timestamp DESC LIMIT ?',
        (N_ROWS,)
    )

    if rows:
        df = pd.DataFrame(rows)
        df['bet_triggered'] = df['bet_triggered'].astype(bool)
        df['score'] = df['score'].round(4)
        print(f'Last {len(df)} runs:')
        print(df.to_string(index=False))

        bets = df['bet_triggered'].sum()
        print(f'\nBets triggered: {bets}/{len(df)} ({100*bets/len(df):.1f}%)')
    else:
        print('No runs in SQLite yet. Start the loop (Section 4) to collect data.')
else:
    print(f'SQLite not found at {SQLITE_DB_FILE}. Start the loop to create it.')

## 8. External Data Status

Check freshness of external data collectors.

In [ ]:
from config import EXTERNAL_DIR
import json
from datetime import datetime

external_files = [
    'fred_snapshots.jsonl',
    'bls_snapshots.jsonl',
    'weather_snapshots.jsonl',
    'odds_snapshots.jsonl',
]

print('═' * 60)
print('  EXTERNAL DATA STATUS')
print('═' * 60)

for fname in external_files:
    fpath = EXTERNAL_DIR / fname
    if fpath.exists():
        # Count lines
        with open(fpath, 'r') as f:
            lines = f.readlines()
        count = len(lines)
        
        # Get last timestamp
        if count > 0:
            last_line = json.loads(lines[-1])
            ts = last_line.get('collected_at', last_line.get('timestamp', 'unknown'))
            print(f'✓ {fname:<30s} {count:>5d} rows  (last: {ts})')
        else:
            print(f'✗ {fname:<30s} empty')
    else:
        print(f'✗ {fname:<30s} not found')

print('═' * 60)